# Inversion-only: bypass the feature-extraction step

If you already have a CSV of morphology features (e.g. from a
different segmenter or a different feature library), feed it
directly into `infer_from_features`. Required columns:

- `spheroid_id`
- `total_area`
- `equivalent_diameter`
- `solidity`
- `perimeter`
- `circularity`

Any other columns are passed through. One row per spheroid (or per
frame; the matcher does not distinguish - if you pass frame-resolved
rows the matcher treats each row as an independent observation and
the posterior summary will be over all rows for that spheroid_id).

In [1]:
import pandas as pd

from cll_cpm_inversion import (
    OPERATIONAL_FEATURES, PARAMS,
    infer_from_features,
    load_synthetic_library,
    load_identifiability,
    load_sobol_indices,
    feature_weights_from_sobol,
)

## Worked example: feed three library samples back in

A useful sanity check: a synthetic vector from the library should
match itself at rank 0 and yield a tight posterior around the truth.

In [2]:
lib = load_synthetic_library()
picks = lib.iloc[[10, 200, 400]].copy()
picks.index = ["lib_010", "lib_200", "lib_400"]
features_df = picks[OPERATIONAL_FEATURES].reset_index().rename(
    columns={"index": "spheroid_id"})
features_df

,spheroid_id,total_area,equivalent_diameter,solidity,perimeter,circularity
0,lib_010,175024.888889,472.068374,0.928203,3015.336037,0.243380
1,lib_200,159417.000000,450.528008,0.992046,1547.378105,0.836695
2,lib_400,134601.000000,413.979596,0.523802,53518.134648,0.000591


In [3]:
summary = infer_from_features(features_df, k=20)
summary.round(3)

,spheroid_id,parameter,median,q05,q95,q25,q75,n_matches,loo_r2,identifiability
0,lib_010,width,16.000,5.950,25.200,10.750,17.000,20,0.466,weakly identifiable
1,lib_010,temp,62.738,45.372,69.012,54.041,67.443,20,0.134,unidentifiable
2,lib_010,lambda,0.254,0.130,1.747,0.242,0.540,20,0.042,unidentifiable
3,lib_010,contact,38.516,6.493,45.636,18.035,39.855,20,0.310,weakly identifiable
4,lib_010,cm_adhesion,18.801,3.105,22.705,14.398,19.184,20,0.613,weakly identifiable
5,lib_010,contact_no,4.000,1.000,6.050,2.750,5.000,20,0.127,unidentifiable
6,lib_010,neighbor,5.000,2.000,6.050,3.000,5.000,20,-0.090,unidentifiable
7,lib_200,width,10.500,4.000,14.250,7.000,12.000,20,0.466,weakly identifiable
8,lib_200,temp,44.773,14.746,74.715,36.361,61.598,20,0.134,unidentifiable
9,lib_200,lambda,0.428,0.102,18.034,0.252,2.065,20,0.042,unidentifiable


Compare posterior median to ground truth:

In [4]:
truth = picks[PARAMS].reset_index().rename(columns={"index": "spheroid_id"})
truth_long = truth.melt(id_vars="spheroid_id",
                        var_name="parameter", value_name="truth")
compare = summary.merge(truth_long, on=["spheroid_id", "parameter"])
compare[["spheroid_id", "parameter", "truth", "median",
         "q05", "q95", "identifiability"]].round(2)

,spheroid_id,parameter,truth,median,q05,q95,identifiability
0,lib_010,width,17.00,16.00,5.95,25.20,weakly identifiable
1,lib_010,temp,46.20,62.74,45.37,69.01,unidentifiable
2,lib_010,lambda,0.25,0.25,0.13,1.75,unidentifiable
3,lib_010,contact,39.86,38.52,6.49,45.64,weakly identifiable
4,lib_010,cm_adhesion,18.80,18.80,3.11,22.71,weakly identifiable
5,lib_010,contact_no,4.00,4.00,1.00,6.05,unidentifiable
6,lib_010,neighbor,5.00,5.00,2.00,6.05,unidentifiable
7,lib_200,width,10.00,10.50,4.00,14.25,weakly identifiable
8,lib_200,temp,57.61,44.77,14.75,74.71,unidentifiable
9,lib_200,lambda,0.30,0.43,0.10,18.03,unidentifiable


Notice that even when feeding the library back into itself, the
unidentifiable parameters (`temp`, `lambda`, `contact_no`,
`neighbor`) are not recovered to their true values - the matcher
averages over the 20 most morphologically-similar synthetic samples,
and those samples have very different values of the unidentifiable
parameters. This is the practical signature of unidentifiability:
the feature-to-parameter mapping does not constrain those
parameters.

## Inspecting the matcher internals

The Sobol indices and per-feature weights that drive the matcher
are all loadable.

In [5]:
load_sobol_indices().head()

,feature,parameter,S1,ST,S1_conf,ST_conf
0,total_area,width,0.398471,0.799781,0.150453,0.222964
1,total_area,temp,0.005158,0.022841,0.019998,0.010396
2,total_area,contact,0.023352,0.068909,0.021310,0.021260
3,total_area,neighbor,-0.003504,0.002518,0.011613,0.000856
4,total_area,contact_no,0.006352,0.126507,0.057148,0.041396


In [6]:
weights = feature_weights_from_sobol()
pd.Series(weights, name="weight").to_frame().round(4)

,weight
total_area,0.2138
equivalent_diameter,0.2251
solidity,0.1966
perimeter,0.1983
circularity,0.1662


`circularity` carries the most weight because its mean total-order
Sobol index across the 7 parameters is the largest in this library.
This is what promotes $J_{cc}$ from unidentifiable (under uniform
weights, R^2 < 0.3) to weakly identifiable.